# The "Cleanup" Notebook
## Retention time: 1 day 

In [0]:
import time

# --- CONFIGURATION ---
VOLUME_PATH = "/Volumes/patient_data/ingestion_patient_vitals/raw_jsons/"
RETENTION_DAYS = 1 
SECONDS_IN_DAY = 86400

# Convert retention days to milliseconds (Databricks uses ms for mod time)
now_ms = time.time() * 1000
retention_ms = RETENTION_DAYS * SECONDS_IN_DAY * 1000
threshold_ms = now_ms - retention_ms

print(f"Cleaning files older than {RETENTION_DAYS} day...")

try:
    files = dbutils.fs.ls(VOLUME_PATH)
    deleted_count = 0
    retained_count = 0

    for file in files:
        # Skip directories, only look at files
        if not file.isDir():
            if file.modificationTime < threshold_ms:
                print(f"Deleting: {file.name} (Modified: {file.modificationTime})")
                dbutils.fs.rm(file.path)
                deleted_count += 1
            else:
                retained_count += 1

    # --- THIS PART GENERATES THE OUTPUT YOU WANT ---
    print("-" * 30)
    print(f"DELETED: {deleted_count} files (older than {RETENTION_DAYS} day)")
    print(f"RETAINED: {retained_count} files (newer than {RETENTION_DAYS} day)")
    print("-" * 30)

except Exception as e:
    print(f"Error accessing Volume: {e}")